# Evaluation Notebook (Parts 1–3)

---


Version: 1.0.3 (<span style="color:orange; text-decoration: underline;">do not modify</span>)

**Copyright (c) 2026 UCD COMP47650**  

- Private coursework for University College Dublin.  
- Do **not** share publicly or upload to repositories.  
- Do **not** submit this notebook to AI tools or external services.

---


# Misc. Setup

In [1]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# Project base dir (notebook folder)
BASE_DIR = Path.cwd().resolve().parent
print(f"Project base directory: {BASE_DIR}")

# Add project root and datasets to sys.path for imports
for path in [BASE_DIR, BASE_DIR / 'datasets']:
    str_path = str(path)
    if str_path not in sys.path:
        sys.path.insert(0, str_path)

# Seed everything for reproducibility
from scripts.utils import seed_all
seed_all(2026)

Project base directory: /Users/ant-smalls/Desktop/UCD-Spring/COMP47650-Deep_Learning/final_assignment/project_final_version/23225831_dl_project



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/Users/ant-smalls/Desktop/UCD-Spring/COMP47650-Deep_Learning/final_assignment/project_final_version/23225831_dl_project/dl_env/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_

---

# 1. Glyph Classification Model Evaluation (Part 1)

- Run the evaluation cell and ensure it executes without errors.

- The output should display the final test accuracy.

- <span style="color:orange; text-decoration: underline;">Do NOT modify this cell code.</span>


In [2]:
# THIS CELL SHOULD EVALUATE WITHOUT ERRORS

# Seed for reproducibility
from scripts.utils import seed_all
seed_all(2026)

import torch
from scripts.part1_preprocessing import part1_build_preprocess_args, part1_preprocess_x, part1_preprocess_y, part1_pad_collate
from models.part1_glyph_model import part1_build_vocab, part1_build_model_args, part1_glyph_classification_model, part1_test_model
from scripts.utils import get_dataloader

checkpoint_path=BASE_DIR / 'checkpoints' / 'part1_glyph_model.pth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Vocab, model and dataset arguments
vocab_obj = part1_build_vocab()
model_args = part1_build_model_args(vocab=vocab_obj)
preprocess_args = part1_build_preprocess_args()
print(f"{model_args = }")
print(f"{preprocess_args = }")

# Instantiate model
model = part1_glyph_classification_model(**model_args).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}","\n"*2)

# DataLoader (test)
H5_FILE = BASE_DIR / 'datasets' / 'glyph_80k.h5'
# H5_FILE = BASE_DIR / 'datasets' / 'glyph_1k.h5'
assert H5_FILE.exists(), f"Dataset file not found: {H5_FILE}"
batch_size = 128

test_loader = get_dataloader(
    H5_FILE,
    batch_size,
    split='test',
    loader_setup = (vocab_obj, preprocess_args, part1_preprocess_x, part1_preprocess_y, part1_pad_collate),
    shuffle=False,
    use_cache=False,
)

# Evaluate model performance on test data
test_acc = part1_test_model(
    model,
    test_loader,
    checkpoint_path,
    device=device,
)

print(
    f"Part1 (Test): Class Acc = {100 * test_acc:.2f}%"
)


model_args = {'input_dim': 386, 'num_classes': 18}
preprocess_args = {'bos_value': 2, 'eos_value': 3, 'pad_value': -5, 'sep_value': -1, 'zero_pad_value': 0, 'pad_token_value': -5, 'vec_length': 386}
Trainable parameters: 77,090 


Processing test batches from: datasets/glyph_80k.h5
Using device: cpu
Model from checkpoint at Epoch 18, (Valid acc=0.9078): checkpoints/part1_glyph_model.pth


Epoch 18 [Test]: 100%|██████████| 125/125 [01:29<00:00,  1.39it/s, Batch Class Acc=0.9007]

Part1 (Test): Class Acc = 90.07%


<span style="color:orange; text-decoration: underline;">**Our benchmark:**</span> *evaluation on test data completed in 9.0s on Apple M chip (using MPS)*

```text
Trainable parameters: 33,490
----------------------------------------------
Part1 (Train): Class Acc = 99.70%
Part1 (Valid): Class Acc = 96.13%
----------------------------------------------
Model from checkpoint at Epoch 17, (Valid acc=0.9613): checkpoints/part1_glyph_model.pth
Epoch 17 [Test]: 100%|██████████| 250/250 [00:09<00:00, 25.50it/s, Batch Class Acc=0.9531]
----------------------------------------------
Part1 (Test): Class Acc = 95.31%
----------------------------------------------
```

---

# 2. Infix Model Evaluation (Part 2)

- Run the evaluation cell and ensure it executes without errors.

- The output should display the final test accuracy.

- <span style="color:orange; text-decoration: underline;">Do NOT modify this cell code.</span>

In [3]:
# THIS CELL SHOULD EVALUATE WITHOUT ERRORS

# Seed for reproducibility
from scripts.utils import seed_all
seed_all(2026)

import torch
from scripts.part2_preprocessing import part2_build_preprocess_args, part2_preprocess_x, part2_preprocess_y, part2_pad_collate
from models.part2_infix_model import part2_build_vocab, part2_build_model_args, part2_infix_recognition_model, part2_test_model
from scripts.utils import get_dataloader

checkpoint_path=BASE_DIR / 'checkpoints' / 'part2_infix_model.pth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Vocab, model and dataset arguments
vocab_obj = part2_build_vocab()
model_args = part2_build_model_args(vocab=vocab_obj)
preprocess_args = part2_build_preprocess_args(vocab_obj)
print(f"{model_args = }")
print(f"{preprocess_args = }")

# Instantiate model
model = part2_infix_recognition_model(**model_args).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}","\n"*2)

# DataLoader (test)
H5_FILE = BASE_DIR / 'datasets' / 'infix_88k.h5'
# H5_FILE = BASE_DIR / 'datasets' / 'infix_1k.h5'
assert H5_FILE.exists(), f"Dataset file not found: {H5_FILE}"
batch_size = 128

test_loader = get_dataloader(
    H5_FILE,
    batch_size,
    split='test',
    loader_setup = (vocab_obj, preprocess_args, part2_preprocess_x, part2_preprocess_y, part2_pad_collate),
    shuffle=False,
    use_cache=False,
)

# Evaluate model performance on test data
la, cer = part2_test_model(
    model,
    test_loader,
    checkpoint_path,
    device=device,
)

print(
    f"Part2 (Test): LA = {100 * la:.2f}%, "
    f"forced CER = {100 * cer:.2f}%"
)

model_args = {'vocab_size': 22, 'max_len': 64, 'pad_id': 1, 'bos_id': 2, 'eos_id': 3}
preprocess_args = {'bos_value': 2, 'eos_value': 3, 'pad_value': -5, 'zero_pad_value': 0, 'pad_token_value': 1, 'vec_length': 128}
Trainable parameters: 210,806 


Processing test batches from: datasets/infix_88k.h5
Using device: cpu
Model from checkpoint at Epoch 17, (Valid acc=0.9618): checkpoints/part2_infix_model.pth


Epoch 17 [Test]: 100%|██████████| 138/138 [08:28<00:00,  3.68s/it, Batch LA=0.9509, Batch CER=0.0368]

Part2 (Test): LA = 95.91%, forced CER = 3.19%


<span style="color:orange; text-decoration: underline;">**Our benchmark:**</span> *evaluation on test data completed in 10.8s on Apple M chip (using MPS)*

```text
Trainable parameters: 179,070
----------------------------------------------
Part2 (Train): LA = 99.95%, forced CER = 0.05%
Part2 (Valid): LA = 96.10%, forced CER = 3.96%
----------------------------------------------
Model from checkpoint at Epoch 30, (Valid acc=0.9587): checkpoints/part2_infix_model.pth
Epoch 30 [Test]: 100%|██████████| 275/275 [00:11<00:00, 23.51it/s, Batch LA=0.9360, Batch CER=0.0659]
----------------------------------------------
Part2 (Test): LA = 95.20%, forced CER = 4.95%
----------------------------------------------
```

---
# 3. Postfix Model Evaluation (Part 3)

- Run the evaluation cell and ensure it executes without errors.

- The output should display the final test accuracy.

- <span style="color:orange; text-decoration: underline;">Do NOT modify the cell code.</span>

In [4]:
# THIS CELL SHOULD EVALUATE WITHOUT ERRORS

# Seed for reproducibility
from scripts.utils import seed_all
seed_all(2026)

import torch
from scripts.part3_preprocessing import part3_build_preprocess_args, part3_preprocess_x, part3_preprocess_y, part3_pad_collate
from models.part3_postfix_model import part3_build_vocab, part3_build_model_args, part3_postfix_recognition_model, part3_test_model
from scripts.utils import get_dataloader

checkpoint_path=BASE_DIR / 'checkpoints' / 'part3_postfix_model.pth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Vocab, model and dataset arguments
vocab_obj = part3_build_vocab()
model_args = part3_build_model_args(vocab=vocab_obj)
preprocess_args = part3_build_preprocess_args(vocab_obj)
print(f"{model_args = }")
print(f"{preprocess_args = }")

# Instantiate model
model = part3_postfix_recognition_model(**model_args).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}","\n"*2)

# DataLoader (test)
H5_FILE = BASE_DIR / 'datasets' / 'postfix_208k.h5'
# H5_FILE = BASE_DIR / 'datasets' / 'postfix_1k.h5'
assert H5_FILE.exists(), f"Dataset file not found: {H5_FILE}"
batch_size = 128

test_loader = get_dataloader(
    H5_FILE,
    batch_size,
    split='test',
    loader_setup = (vocab_obj, preprocess_args, part3_preprocess_x, part3_preprocess_y, part3_pad_collate),
    shuffle=False,
    use_cache=False,
)

# Evaluate model performance on test data
la, cer = part3_test_model(
    model,
    test_loader,
    checkpoint_path,
    device=device,
)

print(
    f"Part3 (Test): LA = {100 * la:.2f}%, "
    f"forced CER = {100 * cer:.2f}%"
)

model_args = {'vocab_size': 23, 'max_len': 64, 'pad_id': 1, 'bos_id': 2, 'eos_id': 3}
preprocess_args = {'bos_value': 2, 'eos_value': 3, 'pad_value': -5, 'zero_pad_value': 0, 'pad_token_value': 1, 'vec_length': 128}
Trainable parameters: 49,015 


Processing test batches from: datasets/postfix_208k.h5
Using device: cpu
Model from checkpoint at Epoch 17, (Valid acc=0.9622): checkpoints/part3_postfix_model.pth


Epoch 17 [Test]: 100%|██████████| 325/325 [24:48<00:00,  4.58s/it, Batch LA=0.5602, Batch CER=0.1274]

Part3 (Test): LA = 55.29%, forced CER = 14.41%


<span style="color:orange; text-decoration: underline;">**Our benchmark:**</span> *evaluation on test data completed in 3m21.8s on Apple M chip (using MPS)*

```text
Trainable parameters: 74,788 
----------------------------------------------
Part3 (Train): LA = 99.96%, forced CER = 0.03%
Part3 (Valid): LA = 97.61%, forced CER = 2.20%
----------------------------------------------
Model from checkpoint at Epoch 10, (Valid acc=0.9773): checkpoints/part3_postfix_model.pth
Epoch 10 [Test]: 100%|██████████| 650/650 [02:59<00:00,  3.63it/s, Batch LA=0.9573, Batch CER=0.0322]
----------------------------------------------
Part3 (Test): LA = 95.69%, forced CER = 3.64%
----------------------------------------------
```